In [0]:
# List of all operational datasets you want to ingest
ops_types = ["flights"]

for ops in ops_types:
    print(f"Processing operational data for: {ops}")
    
    # Dynamic paths matching the Reference Script pattern
    source_path = f"/Volumes/main/lufthansa/landing_zone/ops/{ops}"
    checkpoint_path = f"/Volumes/main/lufthansa/_checkpoints/bronze_ops_{ops}"
    target_table = f"main.lufthansa_bronze.ops_{ops}"

    # Ingestion Logic - Identical behavior to Reference Script
    query = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("multiLine", "true") 
        .load(source_path)
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoint_path}/data")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table))

    # Wait for completion before potentially moving to next ops type
    query.awaitTermination()
    print(f"Finished: {ops}")

In [0]:
# List of all reference datasets you want to ingest
ref_types = ["airports", "countries", "cities", "airlines", "aircraft"]

for ref in ref_types:
    print(f"Processing reference data for: {ref}")
    
    # Dynamic paths based on the current ref type
    source_path = f"/Volumes/main/lufthansa/landing_zone/ref/{ref}"
    checkpoint_path = f"/Volumes/main/lufthansa/_checkpoints/bronze_ref_{ref}"
    target_table = f"main.lufthansa_bronze.ref_{ref}"

    # Ingestion Logic
    query = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("multiLine", "true") 
        .load(source_path)
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoint_path}/data")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table))

    # Wait for this specific reference type to finish before moving to the next
    query.awaitTermination()
    print(f"Finished: {ref}")

In [0]:
# RESET AUTOLOADERS CACHE

checkpoint_path = "/Volumes/main/lufthansa/_checkpoints/bronze_flights"
dbutils.fs.rm(checkpoint_path, recurse=True)

In [0]:
%sql
-- DESCRIBE DETAIL bronze;
-- SELECT * FROM bronze_flights
DROP TABLE main.lufthansa_bronze.ops_flights;

In [0]:
%sql
SELECT * FROM main.lufthansa.bronze_flights

In [0]:
%sql
-- This 'infers' the schema of the string on the fly just for viewing
SELECT schema_of_json(payload) FROM main.lufthansa.bronze_flights LIMIT 1;

In [0]:
%sql
-- This reaches inside the JSON string to grab a specific value
SELECT 
  payload:FlightStatusResource.Flights.Flight[0].OperatingCarrier.FlightNumber AS flight_no,
  payload:FlightStatusResource.Flights.Flight[0].Departure.ActualTimeLocal.DateTime AS departure_time
FROM main.lufthansa.bronze_flights